# Aprendizaje por refuerzo

El **aprendizaje por refuerzo** es una disciplina de la _Inteligencia Artificial_ que trata de conseguir que un agente aprenda a tomar las decisiones de manera autónoma en función de la recompensa que se le otorgue. Dicho de otra forma, el **agente** aprenderá qué **acciones** realizar en función del estado actual del **entorno** y las **recompensas** obtenidas, es decir, a base de ensayo y error, tal y como lo haría un niño pequeño.

El **Aprendizaje por refuerzo** está compuesto por cuatro componentes:

- Las **acciones**: son cada una de las decisiones que se pueden tomar en el dominio de nuestro problema.
- El **entorno**: es el ámbito en el que se define el problema. Puede ser simulado o percibido a través de sensores.
- El **agente**: es la entidad que toma las decisiones (acciones).
- La **recompensa**: es un mecanismo de retroalimentación que le dice al agente qué acciones le han llevado a un éxito o fracaso.

El proceso de aprendizaje se resume en el siguiente esquema:

![](https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/Reinforcement_learning_diagram.svg/250px-Reinforcement_learning_diagram.svg.png)

1. El **agente** decide realizar una **acción** en función del estado actual percibido así como de las **recompensas** obtenidas.
2. La **acción** realizada por el **agente** interactúa con el **entorno**.
3. Se calcula la **recompensa** que debe obtener el **agente** y se le proporciona el nuevo estado del **entorno**.

_Referencias_:

- [Curso completo UC Berkeley CS 285](http://rail.eecs.berkeley.edu/deeprlcourse-fa19/)

## Las bases del aprendizaje por refuerzo

La disciplina del **Aprendizaje por Refuerzo** se basa en tres pilares fundamentales. El más importante es una herramienta que nos permite describir el problema: _Markov Decission Process (MDP)_. Los dos pilares restantes, la _Programación Dinámica_ y los _métodos de Monte Carlo_ son métodos diseñados para resolver MDPs.

### El primer problema

Antes de definir los conceptos anteriores vamos a plantear y resolver nuestro primer problema de **aprendizaje por refuerzo**:

> Trabajamos para una tienda online. La empresa necesita saber cuándo un nuevo desarrollo ha tenido el efecto deseado. También necesita asegurarse de que las tareas de mantenimiento no tienen un impacto sobre el negocio. Vamos a centrarnos en el escenario en el cual tenemos que **decidir el mejor color para un botón: rojo o verde**.

Para resolver este problema siguiendo un enfoque de **RL** vamos a tener que definir los elementos principales: la **recompensa**, las **acciones** y el **entorno**.

#### La recompensa

El resultado de cualquier acción debe ser **medible**. Esto es, precisamente, el objetivo de la **recompensa**: proporcionar evidencia del resultado de una acción, ya sea bueno o malo. En el caso de nuestro problema, la **recompensa** puede ser positiva si se hace click en el botón del color que sea, o si se ha realizado una venta a través del botón de un color determinado.

#### Evaluación de estrategias

Dentro del **entorno**, el **agente** realizará una acción $a$ perteneciente al conjunto $\mathcal{A}$ de acciones posibles:

$$a \in \mathcal{A}$$

En nuestro problema, las **acciones** que puede realizar el **agente** se limitan a mostrar un botón rojo o verde, por lo que

$$\mathcal{A}\doteq \left \{ a_\mathrm{rojo}, a_\mathrm{verde} \right \}$$

Sabemos también que el **entorno** otorga una **recompensa** $r$ al **agente** obtenida del conjunto $\mathcal{R}$ de recompensas posibles, es decir, $r \in \mathcal{R}$. Esto hace que el **entorno** transite hacia un nuevo estado $s \in \mathcal{S}$.

Es importante destacar que todas estas variables pueden ser **estocásticas**, es decir, si el agente realiza la misma acción con el entorno en el mismo estado es posible que la recompensa obtenida o el nuevo estado del entorno sean distintos.

Para saber cuál de los colores es el mejor podemos calcular la recompena media obtenida para cada una de las acciones para un número $N(a)$ de clientes para la acción $a$:

$$r^{\mathrm{avg}}(a) = \frac{1}{N(a)}\sum_{i=1}^{N(a)} r(a)_i = \frac{r_1 + r_2 + \ldots + r_{N(a)}}{N(a)}$$

Esta manera de calcular la recompensa media no es eficiente, pues necesitamos esperar hasta que varios clientes hayan interactuado con todos nuestros botones de colores para poder calcularla. Vamos a transformarla en una media que se puede calcular iterativamente a partir del valor anterior:

$$r^{\mathrm{avg}}_N(a) \leftarrow r^{\mathrm{avg}}_{N-1}(a) + \frac{1}{N} \left ( r_N(a) - r^{\mathrm{avg}}_{N-1}(a) \right )$$

Ya tenemos definido cómo calcular la recompensa y cómo actualizar su valor de forma _online_. El siguiente reto consiste en decidir qué acción realizar en función de estas recompensas.

#### Eligiendo la mejor acción

Llegamos a la principal diferencia entre _Machine Learning_ y _Reinforcement Learning_: la capacidad del agente de tomar el control absoluto sobre las elecciones de las acciones. En el caso de nuestro problema, podemos decidir mostrar cada uno de los botones el 50% de las veces para sumar las recompensas de cada color y así decidir cuál es el mejor botón (la mejor acción). Este enfoque tiene una pega: el 50% de los clientes van a tener un color de botón que **no** será el óptimo, lo que puede traducirse en pérdidas para la empresa.

Una segunda alternativa es adaptar los porcentajes de cada color a las recompensas que se han ido obteniendo anteriormente. Este enfoque se enmarca dentro de los algoritmos [_bandit_](https://en.wikipedia.org/wiki/Multi-armed_bandit). Por tanto necesitamos observar las recompensas de las distintas acciones múltiples veces para saber cuál será la mejor, pero al mismo tiempo también queremos que el agente tome de manera más frecuente aquellas acciones con mejor recompensa. Esto supone buscar el equilibrio entre _exploración_ y _explotación_, respectivamente.

El primer algoritmo que vamos a diseñar busca el equilibrio contemplando la posibilidad de que con probabilidad $\epsilon$ la decisión de la acción a tomar sea completamente aleatoria:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Callable, Tuple, Sequence

# Configuración de estilo para gráficas (opcional, mejora la estética)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

def make_selection(epsilon: float, r_a: Dict[str, float], actions: List[str]) -> str:
    if np.random.random() < epsilon:
        return np.random.choice(actions)
    else:
        best_action = max(r_a, key=r_a.get)
        # Necesitamos romper el empate entre acciones con la misma recompensa máxima
        best_actions = [action for action, reward in r_a.items() if reward == r_a[best_action]]
        return np.random.choice(best_actions)

def epsilon_greedy(epsilon:float, actions: List[str], reward: Callable[[str], int], n_iters: int, optimal_action: str) -> Tuple[Dict[str, float], List[float]]:
    # Inicialización de las recompensas acumuladas de cada acción
    r_a = dict.fromkeys(actions, 0.0)
    # Inicialización de conteos de cada acción
    n_a = dict.fromkeys(actions, 0)

    r_acum = [] # Recompensa acumulada
    optima_acum = [] # Historial de elección óptima

    for i in range(n_iters):
        a = make_selection(epsilon, r_a, actions)
        r = reward(a)
        n_a[a] += 1
        r_a[a] += (1 / n_a[a]) * (r - r_a[a])
        r_acum.append(r_acum[-1] + r if r_acum else r)

        # Registramos 1 si el agente eligió la mejor acción, 0 si no
        optima_acum.append(1 if a == optimal_action else 0)

    return r_a, r_acum, optima_acum

#### Simulando el entorno

Necesitamos una función que otorgue la recompensa a la opción elegida. En nuestro caso, podemos construir una simulación de si el usuario hará click en el botón en función de su color. Vamos a suponer que los clientes **no siempre** hacen click en los botones, y que clickan en un botón rojo un 40% de las veces, y en un botón verde el 25% de las veces.

Por tanto, construimos una función de simulación que tenga en cuenta estas probabilidades para devolver la recompensa en función del color elegido:

In [ ]:
def reward_color(action):
    if action == "Rojo":
        return 1 if np.random.random() <= 0.40 else 0.0
    else:
        return 1 if np.random.random() <= 0.25 else 0.0

Ya podemos ejecutar múltiples experimentos con diferentes epsilons:

In [ ]:
def simular_multiples_agentes(epsilon, actions, reward_func, n_iters, n_runs, optimal_action):
    # Matriz para guardar el historial de cada ejecución
    optimas_por_paso = np.zeros((n_runs, n_iters))
    
    for run in range(n_runs):
        _, _, optimas = epsilon_greedy(epsilon, actions, reward_func, n_iters, optimal_action)
        optimas_por_paso[run] = optimas
        
    # Calculamos el porcentaje medio de acierto en cada paso (iteración)
    porcentaje_optima = optimas_por_paso.mean(axis=0) * 100
    return porcentaje_optima

In [ ]:
n_iters = 250
n_runs = 1000 # Promediamos sobre 500 "tiendas" independientes
optimal_action = "Rojo"

# Simulamos para distintos valores de epsilon
opt_001 = simular_multiples_agentes(0.01, ["Rojo", "Verde"], reward_color, n_iters, n_runs, optimal_action)
opt_01  = simular_multiples_agentes(0.1, ["Rojo", "Verde"], reward_color, n_iters, n_runs, optimal_action)
opt_05  = simular_multiples_agentes(0.5, ["Rojo", "Verde"], reward_color, n_iters, n_runs, optimal_action)
opt_1  = simular_multiples_agentes(1.0, ["Rojo", "Verde"], reward_color, n_iters, n_runs, optimal_action)

# Gráfica
plt.figure(figsize=(10, 6))
plt.plot(opt_001, label='epsilon=0.01 (Poca exploración)')
plt.plot(opt_01, label='epsilon=0.1 (Equilibrio)')
plt.plot(opt_05, label='epsilon=0.5 (Mucha exploración)')
plt.plot(opt_1, label='epsilon=1.0 (Exploración total)')

plt.xlabel('Iteración')
plt.ylabel(r'% de veces que elige la acción óptima')
plt.title('Dilema Exploración vs Explotación: % Elección de la Acción Óptima')
plt.legend()
plt.ylim(0, 100)
plt.grid(True)
plt.show()

## Procesos de decisión de Markov (MDP)

Como hemos visto antes, nuestro **agente** recibe el estado actual $s$ del **entorno** y una **recompensa** $r$. En base a estas dos variables (aleatorias), el **agente** decide realizar una **acción** $a$ que modificará el **entorno**, obteniéndose un nuevo estado $s'$ y recompensa.

Este bucle puede verse como transiciones entre estados y, por tanto, definirse como un proceso de decisión de Markov, ya que **el próximo estado y la recompensa obtenida dependen únicamente del estado anterior y la acción llevada a cabo**. Dentro de los MDP es el modelo de transición quien define la probabilidad de transición entre estados:

$$
p(s', r|s, a)
$$

Como se puede observar, el modelo de transición describe cuál es la dinámica del **entorno** en el cual el **agente** va a realizar sus **acciones**.

### Control de inventario

Para entender este concepto vamos a suponer que queremos resolver un **problema de control de inventario**:

> Supongamos que tenemos una tienda donde solo se vende un único producto. Cada día los clientes realizan compras del producto, por lo que es necesario realizar pedidos al proveedor para no quedarnos sin stock, lo que significaría que estaríamos perdiendo ventas. Tampoco tiene sentido acumular stock indefinidamente, pues necesitaríamos unas instalaciones más grandes que cuestan dinero. ¿Cuándo es el momento óptimo para hacer un restock de inventario?

En primer lugar tenemos que definir los posibles **estados** que podemos observar del **entorno**. En este caso, y por simplicidad, suponemos que como mucho podré almacenar 2 items:

$$
\mathcal{S} = \left \{ 0, 1, 2 \right \}
$$

Con respecto a las **acciones** que el agente puede realizar, tendremos dos posibles acciones:

$$
\mathcal{A} = \left \{ \mathrm{restock}, \mathrm{nada} \right \}
$$

Además de esto, nos faltaría definir las probabilidades de transición entre estados, es decir, el modelo de transición. Por simplicidad, podemos suponer que la probabilidad de una venta individual durante un día es 

$$p(\mathrm{venta})=0.7$$

Por último, tenemos que definir la **recompensa**, que en este caso puede ser simplemente si se realiza venta o no. Esta recompensa está determinada por el hecho de disponer de stock y por la probabilidad de venta definida anteriomente:

$$
f(s,a) = \left\{\begin{matrix}
1 & \text{si } s>0 \text{ y hay venta}\\ 
1 & \text{si } a=\textrm{restock} \text{ y hay venta}\\ 
0 & \text{en otro caso}
\end{matrix}\right.
$$

Sabiendo esto, podemos definir todas las transiciones posibles entre estados mediante una tabla:

| $s$ |         $a$        | $p(s' \mid s, a)$ | $s'$ | $r$ |
|-------|----------------------|-------------------|--------|-------|
| 0     | $\mathrm{nada}$    | $1-p(venta)$    | 0      | 0     |
| 0     | $\mathrm{nada}$    | $p(venta)$      | 0      | 0     |
| 0     | $\mathrm{restock}$ | $1-p(venta)$    | 1      | 0     |
| 0     | $\mathrm{restock}$ | $p(venta)$      | 0      | 1     |
| ...   | ...                  | ...               | ...    | ...   |

También podemos representar las transiciones mediante una matriz de transición de estados:

In [ ]:
p_venta = 0.7

p_none = np.array([
    [1.0,       0.0,            0.0],
    [p_venta,   1 - p_venta,    0.0],
    [0.0,       p_venta,        1 - p_venta]])

p_restock = np.array([
    [p_venta,   1 - p_venta,    0.0],
    [0.0,       p_venta,        1 - p_venta],
    [0.0,       0.0,            1.0]])

print(f"La probabilidad de pasar de no tener a tener stock (1 elemento) sin hacer nada es de {p_none[0][1]:.2f}")
print(f"La probabilidad de pasar de no tener a tener stock (1 elemento) haciendo restock es de {p_restock[0][1]:.2f}")

De la misma forma podemos definir una matriz de recompensas:

In [ ]:
r_none = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0]])

r_restock = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 0]])

Vamos a analizar distintas estrategias (o políticas) de nuestro agente:

- Hacer restock cuando haya hueco en el almacén
- Hacer restock cuando se vacíe el almacén
- Hacer restock aleatoriamente

Recordemos que nuestro agente solo puede realizar dos posibles acciones: $\mathcal{A} = \left \{ \mathrm{restock}, \mathrm{nada} \right \}$. Vamos a codificar las acciones como un enumerado:

In [ ]:
from enum import Enum

class Accion(Enum):
    NADA =      0
    RESTOCK =   1

Codificamos también las tres estrategias como funciones que reciben el estado actual y devuelven el siguiente estado:

In [ ]:
def siempre_lleno(estado_actual: int) -> Enum:
    return Accion.RESTOCK if estado_actual < 2 else Accion.NADA

def nunca_vacio(estado_actual: int) -> Enum:
    return Accion.RESTOCK if estado_actual < 1 else Accion.NADA

def aleatorio(estado_actual: int) -> Enum:
    if estado_actual == 2:
        return Accion.NADA
    if np.random.random() <= 0.5:
        return Accion.NADA
    else:
        return Accion.RESTOCK

Por último, vamos a simular el entorno mediante una función que recibe el estado actual y la acción elegida por el agente para devolver el siguiente estado y la recompensa obtenida:

In [ ]:
transiciones = [p_none, p_restock]
recompensas = [r_none, r_restock]

def entorno(estado_actual: int, accion: Enum) -> Tuple[int, int]:
    # Se obtienen las probabilidades de transitar a nuevos estados en función del actual y la acción elegida
    probs = transiciones[accion.value][estado_actual]

    estado_siguiente = np.random.choice(a = [0, 1, 2], p = probs)

    recompensa = recompensas[accion.value][estado_actual][estado_siguiente]

    return estado_siguiente, recompensa

Vamos a comprobar que la simulación funciona correctamente usando, por ejemplo, la política de mantener el almacén siempre lleno:

In [ ]:
s = 1 # Empezamos con un elemento en el almacén

for i in range(15):
    a = siempre_lleno(s)
    s_p, r = entorno(s, a)

    print(f"Paso: {i:>2}\tEstado inicial: {s}\tAcción: {a.name:>7}\t\tRecompensa: {r}")

    s = s_p

Ahora ya podemos analizar las tres estrategias para ver cuál es la óptima. Para ello simularemos el entorno durante 250 pasos para cada una de las estrategias, y llevaremos un registro de las recompensas obtenidas:

In [ ]:
# Estado inicial: todas las simulaciones comienzan con el almacén vacío
ssl = snv = sa = 0

N_ITERS = 250

r_siempre_lleno = []
r_nunca_vacio = []
r_aleatorio = []
iters = list(range(N_ITERS))

for _ in iters:
    a_siempre_lleno = siempre_lleno(ssl)
    a_nunca_vacio = nunca_vacio(snv)
    a_aleatorio = aleatorio(sa)
    ssl, rsl = entorno(ssl, a_siempre_lleno)
    snv, rnv = entorno(snv, a_nunca_vacio)
    sa, ra = entorno(sa, a_aleatorio)

    r_siempre_lleno.append(rsl)
    r_nunca_vacio.append(rnv)
    r_aleatorio.append(ra)

simulacion = pd.DataFrame({"paso": iters, 
                           "Siempre lleno": np.cumsum(r_siempre_lleno),
                           "Nunca vacío": np.cumsum(r_nunca_vacio),
                           "Aleatorio":np.cumsum(r_aleatorio)})
simulacion = pd.melt(simulacion, id_vars="paso", var_name="Estrategia", value_name="recompensa")

plt.figure(figsize=(10, 6))
sns.lineplot(data=simulacion, x="paso", y="recompensa", hue="Estrategia")
plt.xlabel('Paso')
plt.ylabel('Recompensa acumulada')
plt.title('Comparación de Estrategias de Restock')
plt.legend()
plt.grid(True)
plt.show()


### Políticas y Funciones de Valor

Podemos definir el retorno $G$ como la recompensa total que obtendremos a partir del estado actual hasta el estado final, el cual puede ser inifinito:

$$G\doteq r + r' + r'' + \dotsm + r_T$$

Para evitar el poder explosivo de un estado terminal infinito $T=\infty$ se aplica un factor de descuento $\gamma$ a las recompensas futuras:

$$G\doteq r + \gamma r' + \gamma^2 r'' + \dotsm = \sum_{k=0}^T \gamma^k r_k$$

De esta forma se le va restando importancia a las recompensas conforme más futuras son.

Una vez hemos definido el retorno, podemos definir el objetivo del aprendizaje por refuerzo: **construir el agente que consiga generar el mayor retorno esperado**. Como veremos a continuación, es difícil calcular el retorno esperado cuando dependen de las recompensas futuras que obtendrá el agente. La forma de calcularlo consistirá en iterar y aprender cuáles estados y acciones llevan a las mejores recompensas. Dicho de otra forma, habrá que iterar para encontrar la mejor política.

### Prediciendo recompensas con la función Estado-Valor

Recordemos que una política $\pi$ es la probabilidad de elegir una acción $a$ dado un estado $s$:

$$\pi = p(a|s)$$

La función estado-valor de una política calcula la recompensa esperada de una política $\pi$ empezando en el estado $s$:

$$V_\pi(s) = \mathbb{E}_\pi[G|s] = \mathbb{E}_\pi \left [ \sum_{k=0}^T \gamma^k r_k | s \right ]$$

Como podemos observar, es necesario conocer la política para calcular su recompensa esperada, pero era precisamente esta recompensa esperada la que nos tenía que guiar para obtener la política.

### Un nuevo entorno simplificado: la pasarela

Vamos a suponer que tenemos una pasarela formada por 5 casillas, y que el agente se encuentra en la casilla 0. El objetivo del agente es llegar a la casilla 5, que es donde se encuentra la recompensa. Para ello, el agente puede realizar dos acciones: moverse a la izquierda o a la derecha. Sin embargo, si el agente se cae de la pasarela (moverse a la izquierda desde el 0) alcanza un estado terminal y no se lleva recompensa. Con estas condiciones, podemos definir nuestro entorno de la siguiente forma:

In [ ]:
import numpy as np
from typing import Sequence, Tuple, List

def simular_pasarela(max_iters: int) -> Tuple[int, set]:
    estado = 0
    # Usamos un 'set' (conjunto) para registrar solo la PRIMERA visita a cada estado
    estados_visitados = set()
    
    for _ in range(max_iters):
        estados_visitados.add(estado)
        estado += np.random.choice([-1, 1])
        
        if estado < 0:
            return 0, estados_visitados
        if estado > 3:
            estados_visitados.add(4)
            return 5, estados_visitados
            
    return 0, estados_visitados

def estimar_valor_estado(num_simulaciones: int, max_iters: int) -> List[float]:
    conteo_recompensas = [0.0] * 5
    visitas_totales = [0] * 5  # Cuántas veces hemos visitado cada estado en total
    
    for i in range(num_simulaciones):
        resultado, estados_visitados = simular_pasarela(max_iters)
        
        for estado in estados_visitados:
            conteo_recompensas[estado] += resultado
            visitas_totales[estado] += 1
            
        if (i + 1) % 100 == 0:
            valores_actuales = [
                conteo / visitas if visitas > 0 else 0.0 
                for conteo, visitas in zip(conteo_recompensas, visitas_totales)
            ]
            print(f"Paso {i + 1}: {[round(v, 2) for v in valores_actuales]}")
            
    return [
        conteo / visitas if visitas > 0 else 0.0 
        for conteo, visitas in zip(conteo_recompensas, visitas_totales)
    ]

estimar_valor_estado(num_simulaciones=1000, max_iters=100)

La función de valor $V(s)$ se define como la "recompensa esperada si empezamos en el estado $s$". Una forma muy elegante y directa de enseñarlo es crear una función que acepte el estado inicial como parámetro y simular 1000 partidas independientes desde cada casilla.

In [ ]:
def simular_desde_estado(estado_inicial: int, max_iters: int) -> int:
    estado = estado_inicial
    for _ in range(max_iters):
        if estado < 0:
            return 0
        if estado > 3:
            return 5
        estado += np.random.choice([-1, 1])
    return 0

def estimar_valor_aislado(num_simulaciones: int, max_iters: int) -> List[float]:
    valores = []
    
    for estado_inicial in range(5):
        if estado_inicial == 4:
            valores.append(5.0)  # El estado terminal tiene el valor de su propia recompensa
            continue
            
        recompensa_acumulada = 0
        for _ in range(num_simulaciones):
            recompensa_acumulada += simular_desde_estado(estado_inicial, max_iters)
        valores.append(recompensa_acumulada / num_simulaciones)
        
    return [round(v, 2) for v in valores]

print(f"Valores finales: {estimar_valor_aislado(1000, 100)}")

### La función Acción-Valor

La función estado-valor $V_\pi(s)$ nos dice cómo de bueno es un estado, asumiendo que actuamos según nuestra política (en este caso, movernos aleatoriamente). Pero para tomar decisiones, nos resulta mucho más útil la función acción-valor $Q_\pi(s, a)$. Esta función responde a la pregunta: "Si estoy en el estado $s$ y elijo forzosamente la acción $a$, ¿qué recompensa esperada obtendré si a partir de ese momento sigo mi política habitual?"

$$Q_\pi(s, a) = \mathbb{E}_\pi[G|s,a] = \mathbb{E}_\pi \left [ \sum_{k=0}^T \gamma^k r_k | s,a \right ]$$

En nuestra pasarela de 5 casillas, las acciones posibles son Izquierda (-1) y Derecha (+1). Vamos a estimar el valor $Q$ forzando el primer paso del agente y, a partir de la nueva casilla, dejándolo que termine el episodio moviéndose al azar.

In [ ]:
from typing import Dict

def estimar_funcion_Q(num_simulaciones: int, max_iters: int) -> Dict[int, Dict[str, float]]:
    # Q será un diccionario donde cada estado tiene valores para 'Izquierda' y 'Derecha'
    Q = {s: {'Izquierda': 0.0, 'Derecha': 0.0} for s in range(4)}
    
    acciones_dict = {'Izquierda': -1, 'Derecha': 1}
    
    for estado in range(4): # Solo evaluamos los estados no terminales (0 al 3)
        for nombre_accion, movimiento in acciones_dict.items():
            recompensa_acumulada = 0
            
            for _ in range(num_simulaciones):
                # 1. FORZAMOS EL PRIMER PASO
                estado_siguiente = estado + movimiento
                
                # 2. EVALUAMOS SI EL PRIMER PASO TERMINÓ EL JUEGO
                if estado_siguiente < 0:
                    recompensa = 0
                elif estado_siguiente > 3:
                    recompensa = 5
                else:
                    # 3. SI NO TERMINÓ, DEJAMOS QUE SIGA AL AZAR
                    recompensa = simular_desde_estado(estado_siguiente, max_iters)
                
                recompensa_acumulada += recompensa
                
            # Guardamos la media para este par (estado, accion)
            Q[estado][nombre_accion] = round(recompensa_acumulada / num_simulaciones, 2)           
    return Q

valores_Q = estimar_funcion_Q(num_simulaciones=2000, max_iters=100)

df_q = pd.DataFrame.from_dict(valores_Q, orient='index')
df_q.index.name = 'Estado (s)'
df_q.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']

# Usamos display() para que Jupyter renderice la tabla en HTML
display(df_q)

### Monte Carlo Control: Aprendiendo a jugar

Ya sabemos estimar el valor de las acciones $Q(s, a)$ para una política fija. Pero el objetivo del Aprendizaje por Refuerzo es encontrar la **política óptima**. 

Para ello, vamos a usar **Monte Carlo Control con Epsilon-Greedy**. El algoritmo funciona así:
1. Empezamos con una Tabla Q vacía (llena de ceros).
2. Jugamos un episodio completo. Para elegir las acciones, miramos nuestra Tabla Q y elegimos la mejor opción (explotación), pero de vez en cuando, con una probabilidad $\epsilon$, elegimos una acción al azar (exploración).
3. Al terminar el episodio, miramos la recompensa final y actualizamos los valores de los pares (estado, acción) que hemos visitado.
4. Repetimos esto miles de veces. Poco a poco, la Tabla Q convergerá a los valores óptimos y nuestra política será la mejor posible.

In [ ]:
import numpy as np
import pandas as pd

def elegir_accion_epsilon_greedy(estado: int, Q: dict, epsilon: float) -> str:
    """Elige una acción basándose en la Tabla Q y la probabilidad epsilon."""
    # Exploración: Elegimos al azar
    if np.random.random() < epsilon:
        return np.random.choice(['Izquierda', 'Derecha'])
    
    # Explotación: Elegimos la mejor acción según Q
    q_izq = Q[estado]['Izquierda']
    q_der = Q[estado]['Derecha']
    
    if q_der > q_izq:
        return 'Derecha'
    elif q_izq > q_der:
        return 'Izquierda'
    else:
        # Si hay empate (muy común al principio cuando todo es 0), elegimos al azar
        return np.random.choice(['Izquierda', 'Derecha'])

def monte_carlo_control(num_episodios: int, epsilon: float):
    # Inicializamos la Tabla Q y diccionarios para contar las visitas
    Q = {s: {'Izquierda': 0.0, 'Derecha': 0.0} for s in range(4)}
    visitas = {s: {'Izquierda': 0, 'Derecha': 0} for s in range(4)}
    
    movimientos = {'Izquierda': -1, 'Derecha': 1}
    
    for _ in range(num_episodios):
        episodio = []
        estado = 0  # Empezamos siempre al inicio de la pasarela
        
        # 1. GENERAR UN EPISODIO
        while True:
            accion = elegir_accion_epsilon_greedy(estado, Q, epsilon)
            episodio.append((estado, accion))
            
            estado += movimientos[accion]
            
            # Comprobamos si el episodio termina
            if estado < 0:
                recompensa = 0
                break
            elif estado > 3:
                recompensa = 5
                break
                
        # 2. ACTUALIZAR LA TABLA Q (First-Visit Monte Carlo)
        visitados_en_este_episodio = set()
        for est, acc in episodio:
            if (est, acc) not in visitados_en_este_episodio:
                visitados_en_este_episodio.add((est, acc))
                
                # Fórmula de la media iterativa
                visitas[est][acc] += 1
                N = visitas[est][acc]
                Q[est][acc] += (1 / N) * (recompensa - Q[est][acc])
                
    return Q

# Entrenamos a nuestro agente
Q_optima = monte_carlo_control(num_episodios=5000, epsilon=0.2)

# Mostramos el resultado
df_q_optima = pd.DataFrame.from_dict(Q_optima, orient='index')
df_q_optima.index.name = 'Estado (s)'
df_q_optima.columns = ['Q(s, Izquierda)', 'Q(s, Derecha)']
display(df_q_optima)

# Reto: ¿eres capaz de crear un agente para el Blackjack?

Con la información que hemos visto hasta ahora, ¿crees que serías capaz de crear un agente para el juego del Blackjack? Para ello, lo primero que tenemos que hacer es definir el entorno. En este caso, el entorno se define por las reglas del juego, y el agente tiene que aprender a jugar a través de la experiencia.

Vamos a usar la librería [Gymnasium](https://gymnasium.farama.org) (un _fork_ de la librería homónima de OpenAI) para simular el entorno del [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/). Esta librería nos proporciona una interfaz sencilla para interactuar con el entorno y obtener las recompensas correspondientes.

In [ ]:
import gymnasium as gym
env = gym.make('Blackjack-v1', natural=False, sab=False)
observation, info = env.reset()
print(f"Observación inicial: Sumas {observation[0]} puntos, la carta visible del crupier es {observation[1]}" + (", además, tienes un As usable" if observation[2]==1 else "."))

Voy a usar un agente cuya política es elegir una acción aleatoria siempre (no es nada inteligente).

In [ ]:
observation, info = env.reset()

fin_episodio = False

while not fin_episodio:
    # Elige una acción: 0 = plantarse, 1 = pedir otra carta
    action = env.action_space.sample() # Aquí podríamos usar una estrategia más inteligente, pero por ahora es aleatoria

    # Recompensas: +1 si ganas, -1 si pierdes, 0 si empatas
    observation, recompensa, fin_episodio, truncated, info = env.step(action)

print(f"¡Se acabó la partida! Recompensa: {recompensa}")
env.close()

In [ ]:
# ¡Tu turno!